# **Fine tune with EfficientnetV2L**

In [ ]:
from src.spectograms import SpectogramConfig
from src.image_preprocessor import ImagePreprocessorConfig, ImagePreprocessor
from src.model_trainer import ModelTrainer
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

In [ ]:
config = SpectogramConfig(
    audio_dir="/content/drive/My Drive/data_passerifromes/paths_cantos_passeriformes.csv",
    out_dir="/content/drive/My Drive/images_spectograms/"
)

In [ ]:
spectograms_path = config.spectograms_paths()
spectograms_path.head(3)

In [ ]:
imag_prep_conif = ImagePreprocessorConfig(
    img_size=(128, 256),
    channels=1,
    batch_size=64,
    aug_proba=0.7
)

preprocessor = ImagePreprocessor(config.out_dir)
data = preprocessor.load_data_from_directory("/content/images_spectograms")

#### Check augmentation

In [ ]:
rec = data.sample(1).iloc[0]
rec

In [ ]:
def show_img_stats(img):
    if isinstance(img, tf.Tensor):
        print((img.shape, img.dtype, img.numpy().min(), img.numpy().max()))
    elif isinstance(img, np.array):
        print((img.shape, img.dtype, img.min(), img.max()))
    else:
        print(f"unexpected type: {type(img)}")

img = preprocessor.read_image(rec.image_path)
fig, axs = plt.subplots(3, 4, sharex='all', sharey='all', figsize=(16, 7))
for i, ax in enumerate(axs.flat):
    if i == 0:
        ax.imshow(img, cmap='viridis')
        show_img_stats(img)
    else:
        img1 = preprocessor.augment_image(img)
        ax.imshow(img1, cmap='viridis')
        show_img_stats(img1)
plt.tight_layout()
plt.show()

#### Check dataset

In [ ]:
dev_data = data.sample(500)
dev_ds = preprocessor.create_training_dataset(dev_data)
dev_ds

In [ ]:
elem = next(iter(dev_ds.take(1)))
elem[1]

In [ ]:
fig, axs = plt.subplots(3, 4, sharex='all', sharey='all', figsize=(16, 8))
for i, ax in enumerate(axs.flat):
    img = elem[0][i]
    show_img_stats(img)
    ax.imshow(img, cmap="viridis")
    ax.set_title(f"label:{np.argmax(elem[1][i].numpy())}")
plt.tight_layout()
plt.show()

# Neural network

In [ ]:
model_name = "EfficientNetV2L"
model = ModelTrainer(
    model_name=model_name,
    img_shape=(128, 256, 1),
    n_classes=667,
    dropout_rate=0.2,
    label_smoothing=0.1,
    weights="imagenet" ,
    model_dir="/content/drive/MyDrive/model/weights_{model_name}.weights.h5",
    fine_tune_layers=200
)

In [ ]:
import os
import re
from sklearn.model_selection import train_test_split

# --- Split SIN data leakage: se reparte por GRABACION, no por chunk ---
# En 03_image_spectogram_creation cada imagen se nombra "{stem_audio}_{offset}.jpeg",
# por lo que todos los chunks de una misma grabacion comparten el mismo recording_id.
# Si el split se hiciera a nivel de imagen, chunks del mismo audio caerian en
# train/valid/test a la vez e inflarian artificialmente las metricas.
data["recording_id"] = data["image_path"].apply(
    lambda p: re.sub(r"_\d+\.jpe?g$", "", os.path.basename(p))
)

# Una fila por grabacion (cada grabacion pertenece a una sola especie) para poder
# estratificar por 'label' sin separar los chunks de un mismo audio.
recordings = data.drop_duplicates("recording_id")[["recording_id", "label"]]

train_rec, valid_rec = train_test_split(
    recordings, test_size=0.3, random_state=42, stratify=recordings["label"]
)
valid_rec, test_rec = train_test_split(
    valid_rec, test_size=0.5, random_state=42, stratify=valid_rec["label"]
)

# Propagar el split de cada grabacion a todos sus chunks/imagenes.
train_df = data[data["recording_id"].isin(set(train_rec["recording_id"]))]
valid_df = data[data["recording_id"].isin(set(valid_rec["recording_id"]))]
test_df  = data[data["recording_id"].isin(set(test_rec["recording_id"]))]

# Garantia: ninguna grabacion aparece en mas de un split (no hay leakage).
assert not (set(train_df["recording_id"]) & set(valid_df["recording_id"]))
assert not (set(train_df["recording_id"]) & set(test_df["recording_id"]))
assert not (set(valid_df["recording_id"]) & set(test_df["recording_id"]))

print(f"Split imagenes:    {len(train_df)} vs {len(valid_df)} vs {len(test_df)}")
print(f"Split grabaciones: {len(train_rec)} vs {len(valid_rec)} vs {len(test_rec)}")
print(f"model_name: {model_name}")

In [ ]:
# Crear los tf.data.Dataset a partir de los splits por grabacion.
# train -> con augmentation/shuffle; valid -> sin augmentation (igual que test_ds).
train_ds = preprocessor.create_training_dataset(train_df)
val_ds   = preprocessor.create_validation_dataset(valid_df)

In [ ]:
%%time
model, history = model.train(
    train_dataset=train_ds,
    val_dataset=val_ds,
    learning_rate=1e-4,
    epochs=80
)

In [ ]:
def show_history(history):
    """Show history"""
    history_frame = pd.DataFrame(history.history)
    history_frame.index = pd.RangeIndex(1, len(history_frame) + 1, name="epoch")
    display(history_frame.style\
        .highlight_min(color='lightgreen', subset=['val_loss'])\
        .highlight_max(color='lightgreen', subset=['val_acc'])
    )
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    history_frame.loc[:, ['loss', 'val_loss']].plot(ax=ax[0], title='loss')
    history_frame.loc[:, ['acc', 'val_acc']].plot(ax=ax[1], title='acc')
    plt.tight_layout()
    plt.show()
    
show_history(history)

# Evaluation

In [ ]:
test_ds = preprocessor.create_validation_dataset(test_df)

In [ ]:
true_labels = []
i=0
test_ds_size = test_ds.cardinality().numpy()
print(test_ds_size)
for batch in test_ds:
    _, batch_labels = batch  # assuming that labels are the second element of the batch tuple
    true_labels.extend(batch_labels.numpy().tolist())
    i+=1

In [ ]:
pred_labels = model.predict(test_ds, verbose=1)

In [ ]:
from sklearn.metrics import accuracy_score
true_label = np.array(true_labels)
true_label = np.argmax(true_labels, axis=1)

pred_label=tf.argmax(pred_labels, axis=1).numpy()
# # assume y_true and y_pred are your true and predicted labels, respectively
acc = accuracy_score(true_label, pred_label)
print(acc)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(data.label)

def decode_label(label):
    integer_encoded = label_encoder.transform(label)
    return tf.one_hot(integer_encoded, depth=model.n_classes), integer_encoded

In [ ]:
_, tru_label = decode_label(data[cfg.label].values)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
precision = precision_score(true_label, pred_label, average='macro')
recall = recall_score(true_label, pred_label,  average='macro')
f1_score = f1_score(true_label, pred_label, average='macro')

print("Precision: ", precision)
print("Recall: ", recall)
print("F1_score: ", f1_score)

true_label_str = label_encoder.inverse_transform(true_label)
pred_label_str = label_encoder.inverse_transform(pred_label)

# classification_report
print(classification_report(true_label_str, pred_label_str))

In [ ]:
import pickle
with open(f"/content/drive/MyDrive/model/label_encoder_{model_name}.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [ ]:
test_df.to_csv(f"/content/drive/MyDrive/test_{model_name}.csv")

In [ ]:
#from google.colab import runtime
#runtime.unassign()